# QuantLab · primer modelo para una ronda de práctica

Este notebook descarga datos públicos, entrena exclusivamente con train, selecciona una configuración con validation y genera predictions.csv para live, que no contiene etiquetas.

La modalidad predeterminada usa un **mercado sintético**. Los IDs identifican filas, no revelan la identidad de un activo. Las eras públicas están permutadas: no ordenes temporalmente por su nombre. El score oficial es la media de Spearman por era después del cierre; no mide rentabilidad. No hay premios QP en torneos ML.


In [ ]:
%pip install -q pandas==2.2.2 numpy==1.26.4 scipy==1.17.1 pyarrow==17.0.0 scikit-learn==1.9.0 requests


In [ ]:
import os, io, hashlib
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import HistGradientBoostingRegressor

WORKER_URL = os.environ.get("QUANTLAB_WORKER_URL", "https://quantlab2.onrender.com").rstrip("/")
TOURNAMENT_SLUG = os.environ.get("QUANTLAB_TOURNAMENT_SLUG", "ml-practice")

def get_json(path, **params):
    response = requests.get(WORKER_URL + path, params=params, timeout=90)
    response.raise_for_status()
    return response.json()

tournaments = get_json("/tournament/ml")
tournament = next(t for t in tournaments if t["slug"] == TOURNAMENT_SLUG)
datasets = get_json("/ml/datasets", tournament_id=tournament["id"])["datasets"]
round_number = max(d["round_number"] for d in datasets if d["kind"] == "live")
records = {d["kind"]: d for d in datasets if d["round_number"] == round_number}
assert set(records) == {"train", "validation", "live"}

def read_frame(record):
    url = get_json(f"/ml/datasets/{record['id']}/download", kind=record["kind"])["url"]
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    frame = pd.read_parquet(io.BytesIO(response.content))
    assert len(frame) == record["row_count"] and frame["id"].is_unique
    return frame

train, validation, live = [read_frame(records[k]) for k in ("train", "validation", "live")]
assert "target" in train and "target" in validation and "target" not in live
features = records["train"]["feature_cols"]
assert np.isfinite(train[features + ["target"]].to_numpy()).all()
assert np.isfinite(validation[features + ["target"]].to_numpy()).all()
assert np.isfinite(live[features].to_numpy()).all()
print("Versión:", records["live"]["version"])
print("Cierre UTC:", records["live"]["closes_at"])
print("Filas train / validation / live:", len(train), len(validation), len(live))


## Selección usando solo validation

Se comparan dos configuraciones de HistGradientBoostingRegressor con semilla fija. No se reentrena con validation para mantener exactamente el procedimiento de referencia del piloto técnico. Las variables contienen señal y ruido; ninguna configuración tiene una victoria garantizada.


In [ ]:
def ranked(frame, values):
    ranks = pd.Series(values, index=frame.index).groupby(frame["era"]).rank(pct=True)
    return pd.DataFrame({"id": frame["id"], "prediction": ranks})

def scores_by_era(frame, prediction):
    checked = frame[["id", "era", "target"]].merge(prediction, on="id", validate="one_to_one")
    return pd.Series({
        era: rows["prediction"].corr(rows["target"], method="spearman")
        for era, rows in checked.groupby("era")
    })

best_model, best_score = None, float("-inf")
for leaves in (7, 15):
    model = HistGradientBoostingRegressor(
        max_iter=100, max_leaf_nodes=leaves, learning_rate=0.05,
        l2_regularization=1.0, random_state=42,
    )
    model.fit(train[features], train["target"])
    scores = scores_by_era(validation, ranked(validation, model.predict(validation[features])))
    print({"max_leaf_nodes": leaves, "validation_spearman_mean": scores.mean(), "std": scores.std(ddof=0)})
    if scores.mean() > best_score:
        best_model, best_score = model, scores.mean()
print("Configuración seleccionada:", best_model.max_leaf_nodes)


## Generar el archivo para enviar

Las predicciones se convierten en rangos entre 0 y 1 dentro de cada era. El CSV incluye todos los IDs de live una sola vez. El servidor vuelve a validarlo, guarda un recibo SHA-256 y selecciona el último envío válido antes del plazo. Un archivo inválido no reemplaza el anterior.

El notebook no contiene claves de servicio ni solicita el target privado.


In [ ]:
predictions = ranked(live, best_model.predict(live[features]))
assert predictions["id"].is_unique and set(predictions["id"]) == set(live["id"])
assert predictions["prediction"].between(0, 1).all()
checked = predictions.merge(live[["id", "era"]], on="id", validate="one_to_one")
assert checked.groupby("era")["prediction"].nunique().gt(1).all()
predictions.to_csv("predictions.csv", index=False, encoding="utf-8")
payload = open("predictions.csv", "rb").read()
assert len(payload) <= 10 * 1024 * 1024
print("Filas:", len(predictions))
print("SHA-256 del archivo:", hashlib.sha256(payload).hexdigest())
print("Enviar en:", f"https://quant-lab-nine.vercel.app/app/tournaments/{tournament['id']}")


## Leer el resultado después del cierre

En la pestaña Enviar de QuantLab, cargar predictions.csv. Antes del cierre se verá la confirmación de validación, sin score. Después del cierre, el scheduler puntúa y publica el ranking. Conservar la versión, los parámetros y el archivo permite reproducir la entrega. Una puntuación positiva en datos artificiales no demuestra capacidad de pronosticar mercados reales.

El piloto técnico publicado en /docs#evaluacion corresponde a una ronda separada y una cuenta técnica, identificadas como tales.
